# DSPy Prompt Optimization Template

This notebook demonstrates how to use various DSPy optimizers to automatically improve your prompts.

## Overview

**What is DSPy?**
DSPy is a framework that treats prompts as learnable parameters. Instead of manually tweaking prompts, DSPy optimizers can automatically find better prompts based on your examples and evaluation metrics.

**What this notebook covers:**
1. How to define a baseline signature (prompt template)
2. How to create synthetic training examples
3. How to use different DSPy optimizers
4. How to extract and compare optimized prompts

**Optimizers covered:**
- `BootstrapFewShot` - Learns from examples by creating few-shot demonstrations
- `BootstrapFewShotWithRandomSearch` - Adds random search over instructions
- `MIPRO` - Multi-prompt Instruction Proposal Optimizer (most advanced)

## How to use this template:
1. Replace the example prompt (FactStitching) with your own task
2. Create your own synthetic examples or load real data
3. Define your evaluation metric
4. Run the optimizers
5. Extract and compare the optimized prompts

---

## 1. Setup and Installation

Install required packages and import libraries.

In [ ]:
!pip install -q dspy-ai openai together

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.3/291.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.2 MB/s eta 0:00:00


In [ ]:
import dspy
import json
import re
from typing import List, Dict, Any
from datetime import datetime

print(f"DSPy version: {dspy.__version__}")

DSPy version: 3.1.0


### Helper Function to inspect the optimized prompts

In [ ]:
import dspy

def inspect_optimized_program(program, title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    found = False

    for attr_name in dir(program):
        try:
            attr = getattr(program, attr_name)
        except Exception:
            continue

        # We ONLY care about Predict objects
        if isinstance(attr, dspy.Predict):
            found = True
            print(f"\n🔹 Predictor attribute: program.{attr_name}")

            # Instructions
            sig = getattr(attr, "signature", None)
            if sig and getattr(sig, "instructions", None):
                print("\n--- Instructions ---\n")
                print(sig.instructions)
            else:
                print("\n--- No instructions found ---")

            # Demos
            demos = getattr(attr, "demos", None)
            if demos:
                print(f"\n--- Demos attached ({len(demos)}) ---\n")
                for i, d in enumerate(demos, 1):
                    print(f"[Demo {i}]")
                    for k, v in d.items():
                        print(f"  {k}: {str(v)[:80]}")
            else:
                print("\n--- No demos attached ---")

    if not found:
        print("\n⚠️ No dspy.Predict modules found on this program.")


## 2. Model Configuration

Configure the LLM that will be used for:
- **Generation**: Creating responses based on prompts
- **Optimization**: The optimizers will use this model to generate candidate prompts

**Replace the API keys and model names below with your own.**

In [ ]:
# =============================================================================
# CONFIGURATION: Update these with your API keys and preferred models
# =============================================================================

# Option 1: Using Google Colab userdata
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
except:
    # Option 2: Direct assignment (not recommended for sharing)
    OPENAI_API_KEY = "your-openai-key-here"
    TOGETHER_API_KEY = "your-together-key-here"

# Model selection
GENERATION_MODEL = "gpt-4o-mini"  # Model for generating responses
# Alternative: "together_ai/google/gemma-2-9b-it"

# Configure the language model
lm = dspy.LM(
    model=f"openai/{GENERATION_MODEL}",
    api_key=OPENAI_API_KEY,
    max_tokens=2000,
    temperature=0.7
)

# Set as default
dspy.configure(lm=lm)

print(f"✓ Model configured: {GENERATION_MODEL}")

✓ Model configured: gpt-4o-mini


## 3. Define Your Baseline Prompt (Signature)

A DSPy **Signature** is a prompt template with:
- **System instructions** (the docstring)
- **Input fields** (what the model receives)
- **Output fields** (what the model should produce)

**This is the baseline prompt that optimizers will try to improve.**

### Example: Agricultural Advisory Task (Farmer.CHAT)

Replace this with your own task!

In [ ]:
# =============================================================================
# REPLACE THIS SECTION WITH YOUR OWN TASK
# =============================================================================

# Example system prompt for agricultural advisory
BASELINE_SYSTEM_PROMPT = """You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar, India.

**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region

**RESPONSE GUIDELINES:**

1. **Content Quality:**
 - Address the specific concern directly and practically
 - Provide actionable, region-appropriate advice
 - Include timing considerations based on current stage and season
 - Use local examples, varieties, and practices when relevant

2. **Communication Style:**
 - Warm, professional, and encouraging tone
 - Use simple, conversational language with appropriate cultural context
 - Explain technical concepts in simple terms
 - Avoid overly formal or academic language

3. **Practical Advice:**
 - Focus on low-cost, accessible solutions
 - Consider resource constraints of smallholder farmers
 - Mention local availability of inputs/resources
 - Include preventive measures when relevant

4. **Safety & Credibility:**
 - For chemical inputs: mention general categories rather than specific brands
 - Include safety precautions for handling chemicals/equipment
 - Encourage consultation with local experts for complex issues
 - Reference local agricultural departments or extension services

5. **Conversation Flow:**
 - Build on previous advice from chat history when relevant
 - Don't repeat information already covered
 - Ask clarifying questions if critical information is missing
 - Offer to elaborate on specific aspects if helpful

**RESPONSE FORMAT:**
Provide your complete response as a natural conversation. Structure your advice logically but don't force artificial formatting. Keep responses between 150-300 words.

**AVOID:**
- Generic advice that could apply to any crop/region
- Overly technical jargon without explanation
- Repetitive closing statements or tips
- Specific product recommendations without local context
- Assumptions about farmer's resources or experience level
"""

# Define the signature -> Change this according to the use case
class FactStitching(dspy.Signature):
    """Synthesize structured agricultural facts into a natural, conversational response for a farmer."""

    __doc__ = BASELINE_SYSTEM_PROMPT

    farmer_question: str = dspy.InputField(
        desc="The farmer's question about their agricultural concern"
    )
    agricultural_facts: str = dspy.InputField(
        desc="JSON array of structured facts with category, confidence, and relevance scores"
    )
    crop_context: str = dspy.InputField(
        desc="Context about the crop, season, and location if available"
    )
    response: str = dspy.OutputField(
        desc="A natural, conversational response that synthesizes the facts into helpful farming advice"
    )

print("✓ Baseline signature defined")
print(f"  System prompt length: {len(BASELINE_SYSTEM_PROMPT)} characters")

✓ Baseline signature defined
  System prompt length: 2051 characters


## 4. Create Your Module

A DSPy **Module** wraps your signature into a callable component. Think of it as a function that uses your prompt.

We'll use `ChainOfThought` which automatically adds reasoning steps before generating the final output.

In [ ]:

class FarmerChatStitcher(dspy.Module):
    """Module that converts structured facts into conversational farmer advice."""

    def __init__(self, use_cot=True, for_optimization=False):
        """
        Args:
            use_cot: Whether to use Chain of Thought reasoning
            for_optimization: If True, uses Predict (better for demo attachment)
                            If False, uses ChainOfThought
        """
        super().__init__()

        # For optimization, use Predict so BootstrapFewShot can attach demos properly
        # For inference/baseline, use ChainOfThought for better reasoning
        if for_optimization or not use_cot:
            self.generate_response = dspy.Predict(FactStitching)
        else:
            self.generate_response = dspy.ChainOfThought(FactStitching)

    def forward(self, farmer_question: str, agricultural_facts: str, crop_context: str = ""):
        """Generate a conversational response."""
        result = self.generate_response(
            farmer_question=farmer_question,
            agricultural_facts=agricultural_facts,
            crop_context=crop_context
        )
        return result

# Create baseline module (with ChainOfThought for better quality)
baseline_module = FarmerChatStitcher(use_cot=True, for_optimization=False)

print("✓ FarmerChatStitcher module created")
print("  Baseline uses: ChainOfThought (better reasoning)")
print("  Optimizer will use: Predict (better demo attachment)")

✓ FarmerChatStitcher module created
  Baseline uses: ChainOfThought (better reasoning)
  Optimizer will use: Predict (better demo attachment)


## 5. Create Synthetic Training Examples

DSPy optimizers learn from examples. Each example contains:
- **Inputs**: farmer_question, agricultural_facts, crop_context
- **Expected output**: The ideal response

**For your own task:** Replace these with your domain-specific examples.

In [ ]:
# =============================================================================
# REPLACE WITH YOUR OWN TRAINING EXAMPLES
# =============================================================================

TRAINING_EXAMPLES = [
    {
        "farmer_question": "How can I manage pests in my rice field effectively?",
        "crop_context": "Crop: rice, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Brown planthopper is a common pest in rice fields.", "category": "pest identification", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Neem oil is an effective natural pesticide for rice pests.", "category": "pest control", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Maintaining proper water levels can help reduce pest infestations.", "category": "cultural practice", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "To manage pests like the brown planthopper in your rice field, consider using neem oil as a natural pesticide. Ensure your field has proper water levels to deter pests. Regular monitoring is crucial. For best results, consult local agricultural experts in Bihar for tailored advice. Implementing these steps can help protect your crop and increase yield. Remember, early intervention is key to effective pest management."
    },
    {
        "farmer_question": "What's the best time to irrigate wheat crops?",
        "crop_context": "Crop: wheat, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "First irrigation should be done 20-25 days after sowing.", "category": "irrigation timing", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Wheat requires 4-5 irrigations in Bihar's climate.", "category": "irrigation frequency", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Water depth of 5-7 cm is sufficient for each irrigation.", "category": "water depth", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "For wheat irrigation in Bihar, apply the first watering 20-25 days after sowing. Your crop will need 4-5 irrigations throughout the season. Each time, apply water to a depth of 5-7 cm. This schedule helps develop strong roots and improves yield. Monitor soil moisture and adjust based on rainfall. Consult your local Krishi Vigyan Kendra for specific timing recommendations based on your variety and soil type."
    },
    {
        "farmer_question": "How do I improve soil health for maize?",
        "crop_context": "Crop: maize, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Maize requires well-drained soil with pH 5.8-7.0.", "category": "soil requirements", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Green manures like dhaincha improve soil fertility.", "category": "soil improvement", "confidence": 0.85, "bihar_relevance": 0.9},
            {"content": "Regular soil testing identifies nutrient deficiencies.", "category": "soil testing", "confidence": 0.88, "bihar_relevance": 0.85}
        ], indent=2),
        "response": "Improving soil health for maize starts with ensuring good drainage and a pH between 5.8 and 7.0. Incorporate green manures like dhaincha before planting to boost fertility and soil structure. Regular soil testing helps identify what nutrients your soil needs. Consider adding organic compost to improve soil texture and microbial activity. Rotating maize with legumes naturally replenishes nitrogen. For Bihar-specific recommendations, contact your local agricultural department for soil analysis services."
    },
    {
        "farmer_question": "What fertilizer should I use for potato crops?",
        "crop_context": "Crop: potato, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Potato needs balanced NPK - 120:80:100 kg/hectare recommended.", "category": "fertilizer", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Split application improves nutrient uptake.", "category": "application method", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Organic matter improves potato yield and quality.", "category": "soil amendment", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "For potato cultivation in Bihar, use a balanced NPK fertilizer at 120:80:100 kg per hectare. Apply half the nitrogen and all phosphorus and potassium as basal dose, then top-dress the remaining nitrogen 30 days after planting. This split application ensures better nutrient uptake. Also incorporate well-decomposed organic matter before planting to improve soil structure and yield. A soil test will help you fine-tune these recommendations for your specific field conditions."
    },
    {
        "farmer_question": "When should I harvest my lentil crop?",
        "crop_context": "Crop: lentil, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Lentils are ready when 80% of pods turn brown.", "category": "harvest timing", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Harvesting too early reduces yield and quality.", "category": "harvest impact", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Morning harvest reduces shattering losses.", "category": "harvest practice", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "Harvest your lentils when about 80% of the pods have turned brown - this typically occurs 110-120 days after sowing in Bihar. Harvesting too early will reduce both yield and seed quality. For best results, harvest in the early morning when dew is still present, as this reduces pod shattering and seed loss. After cutting, allow the crop to dry in windrows for a few days before threshing. Store in a dry, well-ventilated place to maintain quality."
    },
    {
        "farmer_question": "How do I control weeds in my vegetable garden?",
        "crop_context": "Crop: vegetables, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Mulching with organic materials suppresses weed growth.", "category": "weed control", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Hand weeding is effective for small areas.", "category": "manual control", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Crop rotation disrupts weed life cycles.", "category": "cultural practice", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "For weed control in your vegetable garden, start with mulching using rice straw or dried leaves - this naturally suppresses weed growth while retaining soil moisture. For small plots, hand weeding every 2-3 weeks is very effective, especially when weeds are young. Practice crop rotation to disrupt weed life cycles. Avoid letting weeds flower and set seed. These chemical-free methods are safe for vegetables and cost-effective for small farms. Your local KVK can demonstrate proper mulching techniques."
    },
    {
        "farmer_question": "What variety of mustard gives good oil yield?",
        "crop_context": "Crop: mustard, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Pusa Mustard 29 has 40-42% oil content.", "category": "variety", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Varuna variety is suitable for Bihar's climate.", "category": "variety", "confidence": 0.85, "bihar_relevance": 0.9},
            {"content": "Proper spacing increases oil percentage.", "category": "cultural practice", "confidence": 0.88, "bihar_relevance": 0.85}
        ], indent=2),
        "response": "For good oil yield in Bihar, consider Pusa Mustard 29, which has 40-42% oil content, or Varuna, which is well-adapted to our climate. Both varieties perform excellently when given proper care. Use recommended spacing of 30x10 cm to maximize oil percentage. Ensure adequate sulfur nutrition, as it directly impacts oil content. Timely sowing in mid-October gives best results. Contact your nearest seed center for certified seeds of these varieties."
    },
    {
        "farmer_question": "How can I protect my sugarcane from drought?",
        "crop_context": "Crop: sugarcane, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Mulching conserves soil moisture during dry periods.", "category": "water conservation", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Drought-tolerant varieties like Co 0238 perform better.", "category": "variety selection", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Drip irrigation saves 40% water compared to flood irrigation.", "category": "irrigation method", "confidence": 0.88, "bihar_relevance": 0.8}
        ], indent=2),
        "response": "To protect sugarcane from drought, apply thick mulch of trash or dry leaves between rows to conserve soil moisture. Consider drought-tolerant varieties like Co 0238 for areas with water scarcity. If possible, install drip irrigation, which saves 40% water while maintaining good yields. During critical growth stages (tillering and grand growth), ensure adequate moisture. Practice wider spacing to reduce competition for water. Your district agriculture office can provide information on subsidy schemes for drip irrigation systems."
    }
]

# Convert to DSPy examples
trainset = []
for ex in TRAINING_EXAMPLES:
    example = dspy.Example(
        farmer_question=ex['farmer_question'],
        agricultural_facts=ex['agricultural_facts'],
        crop_context=ex['crop_context'],
        response=ex['response']
    ).with_inputs('farmer_question', 'agricultural_facts', 'crop_context')
    trainset.append(example)

print(f"✓ Created {len(trainset)} training examples")
print(f"\nExample 1:")
print(f"  Q: {trainset[0].farmer_question}")
print(f"  A: {trainset[0].response[:100]}...")

✓ Created 8 training examples

Example 1:
  Q: How can I manage pests in my rice field effectively?
  A: To manage pests like the brown planthopper in your rice field, consider using neem oil as a natural ...


## 6. Define Evaluation Metric - 6-Dimension Quality Assessment

The metric tells DSPy what "good" looks like. We use a comprehensive **ConversationalityEvaluator** that assesses responses across 6 dimensions:

1. **Content Quality** - Direct, practical, actionable advice
2. **Communication Style** - Warm, simple, culturally appropriate
3. **Practical Advice** - Low-cost, accessible solutions
4. **Safety & Credibility** - Safe practices, expert consultation
5. **Conversation Flow** - No repetition, good engagement
6. **Response Format** - Natural conversation, 150-300 words

The evaluator uses an LLM (GPT-4o by default) to score each dimension on a 1-5 scale, then returns an overall score.

**For your task:** You can customize the `EVAL_PROMPT_TEMPLATE` to match your evaluation criteria, or replace with a simpler metric if preferred.

In [ ]:
# =============================================================================
# CELL: Define Evaluation Metric - ConversationalityEvaluator
# =============================================================================

class ConversationalityEvaluator:
    """Evaluates responses using the 6-dimension Farmer.CHAT guidelines"""

    EVAL_PROMPT_TEMPLATE = '''You are an expert evaluator assessing agricultural advisory responses for farmers in Bihar, India.

**EVALUATION STANDARD: Farmer.CHAT Guidelines**

You are evaluating a response from Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar.

**RESPONSE GUIDELINES TO EVALUATE AGAINST:**

1. **Content Quality:** Address the specific concern directly and practically. Provide actionable, region-appropriate advice. Include timing considerations. Use local examples.

2. **Communication Style:** Warm, professional, encouraging tone. Simple, conversational language. Explain technical concepts simply.

3. **Practical Advice:** Focus on low-cost, accessible solutions. Consider resource constraints of smallholder farmers. Mention local availability.

4. **Safety & Credibility:** Mention general categories for chemicals (not specific brands). Include safety precautions. Encourage expert consultation.

5. **Conversation Flow:** Don't repeat information. Ask clarifying questions if needed. Offer to elaborate.

6. **Response Format:** Natural conversation (not bullet points). Logical structure. 150-300 words length.

---

**Question:** {question}

**Response to Evaluate:**
{response}

---

**Rate this response on 6 dimensions (1-5 scale).**

Output ONLY a JSON object with this exact structure (no other text):

{{
   "content_quality": {{
       "score": <number 1-5>,
       "justification": "<one sentence>"
   }},
   "communication_style": {{
       "score": <number 1-5>,
       "justification": "<one sentence>"
   }},
   "practical_advice": {{
       "score": <number 1-5>,
       "justification": "<one sentence>"
   }},
   "safety_credibility": {{
       "score": <number 1-5>,
       "justification": "<one sentence>"
   }},
   "conversation_flow": {{
       "score": <number 1-5>,
       "justification": "<one sentence>"
   }},
   "response_format": {{
       "score": <number 1-5>,
       "justification": "<one sentence>"
   }},
   "overall_score": <average of all 6 scores rounded to 1 decimal>,
   "overall_assessment": "<one sentence summary>"
}}'''

    def __init__(self, eval_lm=None):
        # Use GPT for evaluation (you can change this)
        if eval_lm is None:
            eval_lm = dspy.LM(
                model="openai/gpt-4o",
                api_key=OPENAI_API_KEY,
                max_tokens=3000,
                temperature=0.0
            )
        self.eval_lm = eval_lm

    def evaluate(self, question: str, response: str) -> Dict[str, Any]:
        """Evaluate a response and return scores"""

        prompt = self.EVAL_PROMPT_TEMPLATE.format(
            question=question,
            response=response
        )

        # Use the evaluation LM
        try:
            with dspy.context(lm=self.eval_lm):
                result = self.eval_lm(prompt)

            # Extract the response text
            if isinstance(result, list):
                result_text = result[0] if result else ""
            else:
                result_text = str(result)

            # Parse JSON from response
            json_match = re.search(r'\{[\s\S]*\}', result_text)
            if json_match:
                eval_result = json.loads(json_match.group())

                # Calculate overall if not present
                if 'overall_score' not in eval_result:
                    scores = []
                    for key in ['content_quality', 'communication_style', 'practical_advice',
                               'safety_credibility', 'conversation_flow', 'response_format']:
                        if key in eval_result and 'score' in eval_result[key]:
                            scores.append(eval_result[key]['score'])
                    if scores:
                        eval_result['overall_score'] = round(sum(scores) / len(scores), 1)

                return eval_result

        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
        except Exception as e:
            print(f"Evaluation error: {e}")

        # Return default scores if parsing fails
        return {
            "content_quality": {"score": 3, "justification": "Parsing failed"},
            "communication_style": {"score": 3, "justification": "Parsing failed"},
            "practical_advice": {"score": 3, "justification": "Parsing failed"},
            "safety_credibility": {"score": 3, "justification": "Parsing failed"},
            "conversation_flow": {"score": 3, "justification": "Parsing failed"},
            "response_format": {"score": 3, "justification": "Parsing failed"},
            "overall_score": 3.0,
            "overall_assessment": "Evaluation parsing failed"
        }

    def get_overall_score(self, question: str, response: str) -> float:
        """Get just the overall score (for optimization)"""
        eval_result = self.evaluate(question, response)
        return eval_result.get("overall_score", 3.0)

# Initialize evaluator
evaluator = ConversationalityEvaluator()
print("✓ ConversationalityEvaluator initialized")
print("  Uses 6-dimension evaluation: Content Quality, Communication Style,")
print("  Practical Advice, Safety & Credibility, Conversation Flow, Response Format")


# =============================================================================
# CELL: Define DSPy Metric Function (Boolean for Bootstrap)
# =============================================================================

def conversationality_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> bool:
    """
    DSPy metric function that evaluates response quality using ConversationalityEvaluator.

    Returns True if the response passes quality threshold, False otherwise.
    This boolean return is optimal for BootstrapFewShot optimization.
    """

    # Get the response from prediction
    response = getattr(prediction, 'response', '')
    if not response:
        return False

    # Get the question from example
    question = example.farmer_question

    # Evaluate using our evaluator
    try:
        score = evaluator.get_overall_score(question, response)

        # Define quality threshold (4.0/5.0 = 80% quality)
        QUALITY_THRESHOLD = 4.0

        # Return True if response is high quality
        is_good = score >= QUALITY_THRESHOLD

        if trace:
            normalized = (score - 1) / 4
            status = "✓ PASS" if is_good else "✗ FAIL"
            print(f"  Score: {score:.2f}/5.0 (normalized: {normalized:.3f}) → {status}")

        return is_good
    except Exception as e:
        print(f"Evaluation error: {e}")
        return False  # Fail on error

# Test the metric
print("\nTesting metric on first training example...")
test_pred = baseline_module(
    farmer_question=trainset[0].farmer_question,
    agricultural_facts=trainset[0].agricultural_facts,
    crop_context=trainset[0].crop_context
)

test_result = conversationality_metric(trainset[0], test_pred, trace=True)
print(f"\n✓ Boolean evaluation metric defined and tested")
print(f"  Result: {'PASS ✓' if test_result else 'FAIL ✗'}")
print(f"\nResponse preview:")
print(f"  {test_pred.response[:200]}...")

✓ ConversationalityEvaluator initialized
  Uses 6-dimension evaluation: Content Quality, Communication Style,
  Practical Advice, Safety & Credibility, Conversation Flow, Response Format

Testing metric on first training example...


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


  Score: 4.50/5.0 (normalized: 0.875) → ✓ PASS

✓ Boolean evaluation metric defined and tested
  Result: PASS ✓

Response preview:
  To manage pests in your rice field effectively, start by identifying the brown planthopper, a common pest that can harm your crops. One effective method to control this pest is by using neem oil, whic...


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='```json\...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


╔════════════════════════════════════════════════════════════════════════════╗

║  IMPORTANT: How DSPy Optimizers Store Results                              ║
╔════════════════════════════════════════════════════════════════════════════╗

DSPy optimizers have different behaviors:

✓ BootstrapFewShot:
  - Directly applies optimizations to returned module
  - Standard methods work (.demos, .signature.instructions)
  
⚠ RandomSearch & MIPRO:
  - Store optimizations in 'candidate_programs' metadata
  - Require extraction to access optimized version
  - This is a DSPy framework limitation, not a bug in this template
  
🔍 To verify what's actually sent to the LLM:
  - Run: dspy.inspect_history(n=1) after model execution
  - Or: optimized_model.save("model.json") to export everything

This template handles both behaviors automatically.
╚════════════════════════════════════════════════════════════════════════════╝

## 7. Optimizer #1: BootstrapFewShot

**What it does:**
- Automatically selects the best training examples to use as few-shot demonstrations
- Adds these examples to the prompt so the model can learn from them

**Best for:**
- Tasks where showing examples helps the model understand the pattern
- When you have good training examples

**Parameters:**
- `metric`: The evaluation function
- `max_bootstrapped_demos`: How many examples to include (typically 3-8)
- `max_labeled_demos`: Max examples to try before selecting the best ones

In [ ]:
# =============================================================================
# OPTIMIZER 1: BootstrapFewShot
# =============================================================================

from dspy.teleprompt import BootstrapFewShot

print("\n" + "="*80)
print("OPTIMIZER 1: BootstrapFewShot")
print("="*80)

# Configure optimizer
bootstrap_optimizer = BootstrapFewShot(
    metric=conversationality_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8
)

print("\nCreating student module...")
student_module = FarmerChatStitcher(use_cot=False, for_optimization=True)

print("\nOptimizing...")
optimized_bootstrap = bootstrap_optimizer.compile(
    student=student_module,
    trainset=trainset
)

print("\n✓ BootstrapFewShot complete!")

# Check demos
demos = getattr(optimized_bootstrap.generate_response, 'demos', [])
print(f"  Demos: {len(demos)}")

if len(demos) > 0:
    print(f"  ✓ Successfully added {len(demos)} demonstrations")
    print(f"\n  Demo preview:")
    for i, demo in enumerate(demos, 1):
        print(f"    {i}. Q: {demo.farmer_question[:70]}...")
        if hasattr(demo, 'response'):
            print(f"       A: {demo.response}...")

print("\n" + "="*80)


OPTIMIZER 1: BootstrapFewShot

Creating student module...

Optimizing...


 12%|█▎        | 1/8 [00:05<00:39,  5.65s/it]

  Score: 4.30/5.0 (normalized: 0.825) → ✓ PASS


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content="[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{\n   "c...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Exp

  Score: 4.20/5.0 (normalized: 0.800) → ✓ PASS


 38%|███▊      | 3/8 [00:18<00:31,  6.29s/it]

  Score: 4.00/5.0 (normalized: 0.750) → ✓ PASS


 50%|█████     | 4/8 [00:25<00:25,  6.45s/it]

  Score: 4.50/5.0 (normalized: 0.875) → ✓ PASS
Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.

✓ BootstrapFewShot complete!
  Demos: 8
  ✓ Successfully added 8 demonstrations

  Demo preview:
    1. Q: How can I manage pests in my rice field effectively?...
       A: To effectively manage pests in your rice field, keep an eye out for the brown planthopper, which is quite common. Maintaining proper water levels can significantly help reduce pest infestations, so ensure your field is consistently flooded as per the growth stage of your rice. For pest control, neem oil is an excellent natural pesticide. You can mix it with water and spray it on the plants to deter pests without harming beneficial insects. Regular monitoring and adopting these practices will help keep your rice healthy and productive....
    2. Q: What's the best time to irrigate wheat crops?...
       A: The best time to irrigate your wheat crops in Bihar is to start the first irr

## 8. Optimizer #2: BootstrapFewShotWithRandomSearch

**What it does:**
- Similar to BootstrapFewShot, but also tries variations of the instruction text
- Tests different phrasings and selects the best combination

**Best for:**
- When you want to explore alternative wordings of your prompt
- Tasks where instruction phrasing matters

**Parameters:**
- `metric`: The evaluation function
- `max_bootstrapped_demos`: Number of examples to include
- `num_candidate_programs`: How many prompt variations to try
- `num_threads`: Parallel threads for faster optimization

In [ ]:
# =============================================================================
# OPTIMIZER 2: BootstrapFewShotWithRandomSearch
# =============================================================================

from dspy.teleprompt import BootstrapFewShotWithRandomSearch

print("\n" + "="*80)
print("OPTIMIZER 2: BootstrapFewShotWithRandomSearch")
print("="*80)

# Configure optimizer
random_search_optimizer = BootstrapFewShotWithRandomSearch(
    metric=conversationality_metric,
    max_bootstrapped_demos=4,
    num_candidate_programs=3,
    num_threads=1
)

print("\nCreating student module...")
student_module = FarmerChatStitcher(use_cot=True, for_optimization=False)

print("\nOptimizing... (this may take 2-3 minutes)")
random_search_result = random_search_optimizer.compile(
    student=student_module,
    trainset=trainset,
    valset=trainset[:3]
)

print("\n" + "="*80)
print("EXTRACTING BEST CANDIDATE FROM RANDOMSEARCH")
print("="*80)

# Check for candidates
if hasattr(random_search_result, 'candidate_programs') and random_search_result.candidate_programs:
    print(f"\n✓ RandomSearch stored {len(random_search_result.candidate_programs)} candidate programs")

    # Search for best candidate
    best_candidate = None
    best_candidate_idx = None
    best_score = (-1, False)  # (num_demos, instruction_optimized)

    for i, candidate_dict in enumerate(random_search_result.candidate_programs):
        if isinstance(candidate_dict, dict) and 'program' in candidate_dict:
            candidate_prog = candidate_dict['program']

            if hasattr(candidate_prog, 'generate_response'):
                cand_predictor = candidate_prog.generate_response
                cand_demos = getattr(cand_predictor, 'demos', [])

                # Check instruction
                cand_sig = getattr(cand_predictor, 'signature', None)
                cand_instr = getattr(cand_sig, 'instructions', '') if cand_sig else ''
                instr_different = cand_instr.strip() != BASELINE_SYSTEM_PROMPT.strip()

                if len(cand_demos) > 0 or instr_different:
                    candidate_score = (len(cand_demos), instr_different)
                    print(f"  Candidate {i}: {len(cand_demos)} demos, instruction {'optimized' if instr_different else 'baseline'} (score: {candidate_score})")

                    if candidate_score > best_score:
                        best_score = candidate_score
                        best_candidate = candidate_prog
                        best_candidate_idx = i

    if best_candidate:
        print(f"\n✓ Using candidate {best_candidate_idx} (best: {best_score[0]} demos, instruction {'optimized' if best_score[1] else 'baseline'})")
        optimized_random_search = best_candidate
    else:
        print(f"\n⚠️  No optimized candidates found, using result as-is")
        optimized_random_search = random_search_result
else:
    print("\n⚠️  No candidate_programs found, using result as-is")
    optimized_random_search = random_search_result

# Check final result
print("\n" + "="*80)
print("RANDOMSEARCH OPTIMIZATION RESULTS")
print("="*80)

predictor = optimized_random_search.generate_response
demos = getattr(predictor, 'demos', [])

print(f"\n📚 Few-Shot Demos: {len(demos)}")

if len(demos) > 0:
    print(f"  ✓ Successfully attached {len(demos)} demonstration examples")
    print(f"\n  Demo preview:")
    for i, demo in enumerate(demos[:2], 1):
        print(f"    {i}. Q: {demo.farmer_question[:70]}...")
        if hasattr(demo, 'response'):
            print(f"       A: {demo.response[:70]}...")
else:
    print(f"  ℹ️  No demos attached (RandomSearch optimized instruction only)") # If seeing 0 demos, Adding demos did not increase conversationality_metric, Instruction rewrite alone already hit 100%, DSPy prefers simpler programs if score is identical

# Check instructions - SHOW THE ACTUAL INSTRUCTION
if hasattr(predictor, 'signature'):
    # Try both locations for instruction
    sig = predictor.signature

    # For ChainOfThought, check inner predictor
    if hasattr(predictor, 'predictors'):
        inner_preds = list(predictor.predictors())
        if inner_preds:
            sig = inner_preds[0].signature

    instructions = None
    if hasattr(sig, 'instructions'):
        instructions = sig.instructions
    elif hasattr(sig, '__doc__'):
        instructions = sig.__doc__

    if instructions:
        print(f"\n📝 Instructions: {len(instructions)} characters")

        if instructions.strip() == BASELINE_SYSTEM_PROMPT.strip():
            print(f"  ⚠️  Instructions IDENTICAL to baseline")
        else:
            print(f"  ✓ Instructions OPTIMIZED (different from baseline)")
            print(f"\n  Optimized instruction preview:")
            print(f"  {'─'*80}")
            print(f"  {instructions[:300]}...")
            print(f"  {'─'*80}")
            print(f"\n  Baseline was {len(BASELINE_SYSTEM_PROMPT)} chars, optimized is {len(instructions)} chars")
            print(f"  Reduction: {len(BASELINE_SYSTEM_PROMPT) - len(instructions)} characters")

print("\n✓ RandomSearch optimization complete!")
print("\n" + "="*80)



OPTIMIZER 2: BootstrapFewShotWithRandomSearch
Going to sample between 1 and 4 traces per predictor.
Will attempt to bootstrap 3 candidate sets.

Creating student module...

Optimizing... (this may take 2-3 minutes)
Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:21<00:00,  7.01s/it]

2026/01/10 17:40:59 INFO dspy.evaluate.evaluate: Average Metric: 3 / 3 (100.0%)



New best score: 100.0 for seed -3
Scores so far: [100.0]
Best score so far: 100.0
Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:21<00:00,  7.18s/it]

2026/01/10 17:41:20 INFO dspy.evaluate.evaluate: Average Metric: 1 / 3 (33.3%)



Scores so far: [100.0, 33.33]
Best score so far: 100.0


...


Scores so far: [100.0, 33.33, 100.0, 100.0, 66.67]
Best score so far: 100.0


 12%|█▎        | 1/8 [00:07<00:52,  7.50s/it]


  Score: 4.20/5.0 (normalized: 0.800) → ✓ PASS
Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:22<00:00,  7.45s/it]

2026/01/10 17:44:27 INFO dspy.evaluate.evaluate: Average Metric: 2 / 3 (66.7%)



Scores so far: [100.0, 33.33, 100.0, 100.0, 66.67, 66.67]
Best score so far: 100.0
6 candidate programs found.

EXTRACTING BEST CANDIDATE FROM RANDOMSEARCH

✓ RandomSearch stored 6 candidate programs
  Candidate 0: 0 demos, instruction optimized (score: (0, True))
  Candidate 1: 0 demos, instruction optimized (score: (0, True))
  Candidate 2: 0 demos, instruction optimized (score: (0, True))
  Candidate 3: 0 demos, instruction optimized (score: (0, True))
  Candidate 4: 0 demos, instruction optimized (score: (0, True))
  Candidate 5: 0 demos, instruction optimized (score: (0, True))

✓ Using candidate 0 (best: 0 demos, instruction optimized)

RANDOMSEARCH OPTIMIZATION RESULTS

📚 Few-Shot Demos: 0
  ℹ️  No demos attached (RandomSearch optimized instruction only)

✓ RandomSearch optimization complete!



## 9. Optimizer #3: MIPRO (Optional): Most Advanced, May Not Work with ChainOfThought

**What it does:**
- Multi-prompt Instruction Proposal Optimizer
- Uses an LLM to propose new instruction variations
- Intelligently explores the space of possible prompts
- Combines instruction optimization with example selection

**Best for:**
- Maximum performance gains
- When you have computational budget
- Complex tasks where prompt wording significantly impacts quality

**Parameters:**
- `metric`: The evaluation function
- `num_candidates`: How many instruction variations to propose
- `init_temperature`: Creativity in generating variations (0.0-1.0)

**Note:** MIPRO is the most computationally expensive, most sophisticated optimizer of DSPy and often gives the best results. But I observed compatibility issues with ChainOfThought.

**Use MIPRO only if:**
- You're using `dspy.Predict` (not `ChainOfThought`)
- You're comfortable debugging optimizer issues
- You want to experiment with instruction variations

**For most users: Skip this and use BootstrapFewShot instead.**

In [ ]:
# =============================================================================
# OPTIMIZER 3: MIPROv2 (FINAL SIMPLIFIED VERSION)
# =============================================================================

from dspy.teleprompt.mipro_optimizer_v2 import MIPROv2

print("\n" + "="*80)
print("OPTIMIZER 3: MIPROv2 (Advanced Prompt & Demo Optimization)")
print("="*80)

# Create optimizer with light settings for faster results
mipro_optimizer = MIPROv2(
    metric=conversationality_metric,
    auto="light",  # Options: "light", "medium", "heavy"
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
    verbose=True
)

print("\nCreating student module for MIPRO...")
print("⚠️  Using Predict (not ChainOfThought) for reliable optimization")
mipro_student = FarmerChatStitcher(use_cot=False, for_optimization=True)

# Split data for MIPRO
train_subset = trainset[:6]
val_subset = trainset[6:] if len(trainset) > 6 else trainset[:2]

print(f"  Training: {len(train_subset)} examples")
print(f"  Validation: {len(val_subset)} examples")
print("\nOptimizing... (3-5 minutes)")

try:
    # Run MIPRO optimization
    # MIPRO returns the BEST program directly
    optimized_mipro = mipro_optimizer.compile(
        student=mipro_student,
        trainset=train_subset,
        valset=val_subset,
        minibatch=True
    )

    print("\n" + "="*80)
    print("MIPRO OPTIMIZATION RESULTS")
    print("="*80)

    # Show optimization metadata
    if hasattr(optimized_mipro, 'score'):
        print(f"\n✓ Best score: {optimized_mipro.score}")

    if hasattr(optimized_mipro, 'candidate_programs'):
        print(f"✓ Evaluated {len(optimized_mipro.candidate_programs)} candidate programs")

    # Extract the predictor
    predictor = optimized_mipro.generate_response
    print(f"\nPredictor type: {type(predictor).__name__}")

    # Check demos
    demos = getattr(predictor, 'demos', [])
    print(f"\n📚 Few-Shot Demos: {len(demos)}")

    # If no demos in the main result, search candidate_programs
    if len(demos) == 0 and hasattr(optimized_mipro, 'candidate_programs'):
        print(f"\n  ⚠️  No demos in main result, searching {len(optimized_mipro.candidate_programs)} candidates...")

        best_candidate = None
        best_candidate_idx = None

        for i, candidate_dict in enumerate(optimized_mipro.candidate_programs):
            if isinstance(candidate_dict, dict) and 'program' in candidate_dict:
                candidate_prog = candidate_dict['program']

                # Check if this candidate has demos or optimized instruction
                if hasattr(candidate_prog, 'generate_response'):
                    cand_predictor = candidate_prog.generate_response
                    cand_demos = getattr(cand_predictor, 'demos', [])

                    # Check instruction difference
                    cand_sig = getattr(cand_predictor, 'signature', None)
                    cand_instr = getattr(cand_sig, 'instructions', '') if cand_sig else ''
                    instr_different = cand_instr.strip() != BASELINE_SYSTEM_PROMPT.strip()

                    if len(cand_demos) > 0 or instr_different:
                        print(f"    Candidate {i}: {len(cand_demos)} demos, instruction {'optimized' if instr_different else 'baseline'}")

                        # Prefer candidates with demos
                        if best_candidate is None or len(cand_demos) > len(getattr(best_candidate.generate_response, 'demos', [])):
                            best_candidate = candidate_prog
                            best_candidate_idx = i

        # Use the best candidate we found
        if best_candidate:
            print(f"\n  ✓ Using candidate {best_candidate_idx} instead of main result")
            optimized_mipro = best_candidate
            predictor = optimized_mipro.generate_response
            demos = getattr(predictor, 'demos', [])
            print(f"  ✓ Now have {len(demos)} demos")

    # Display demo info
    if len(demos) > 0:
        print(f"  ✓ Successfully attached {len(demos)} demonstration examples")
        print(f"\n  Demo preview:")
        for i, demo in enumerate(demos[:2], 1):
            print(f"    {i}. Q: {demo.farmer_question[:70]}...")
            if hasattr(demo, 'response'):
                print(f"       A: {demo.response[:70]}...")
    else:
        print(f"  ℹ️  No demos attached (MIPRO selected zero-shot configuration)")

    # Check instructions
    if hasattr(predictor, 'signature') and hasattr(predictor.signature, 'instructions'):
        instructions = predictor.signature.instructions
        print(f"\n📝 Instructions: {len(instructions)} characters")

        # Compare with baseline
        if instructions.strip() == BASELINE_SYSTEM_PROMPT.strip():
            print(f"  ℹ️  Instructions same as baseline (baseline was already optimal)")
        else:
            print(f"  ✓ Instructions optimized (different from baseline)")
            print(f"\n  Optimized instruction preview:")
            print(f"  {instructions[:150]}...")

    # Test the optimized model
    print("\n" + "="*80)
    print("TESTING MIPRO MODEL")
    print("="*80)

    test_example = trainset[3] if len(trainset) > 3 else trainset[0]
    print(f"\nTest question: {test_example.farmer_question[:60]}...")

    mipro_pred = optimized_mipro(
        farmer_question=test_example.farmer_question,
        agricultural_facts=test_example.agricultural_facts,
        crop_context=test_example.crop_context
    )

    print(f"\nMIPRO Response:")
    print(f"{mipro_pred.response[:200]}...")

    # Evaluate
    mipro_score = conversationality_metric(test_example, mipro_pred, trace=True)
    print(f"\nResult: {'PASS ✓' if mipro_score else 'FAIL ✗'}")

    print("\n✓ MIPROv2 optimization complete!")

except Exception as e:
    print(f"\n⚠️  MIPRO optimization failed: {e}")
    print("\nFull error trace:")
    import traceback
    traceback.print_exc()
    print("\n  Using RandomSearch result as fallback...")
    optimized_mipro = optimized_random_search

print("\n" + "="*80)

2026/01/10 17:44:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 2

2026/01/10 17:44:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/01/10 17:44:27 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/01/10 17:44:27 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...



OPTIMIZER 3: MIPROv2 (Advanced Prompt & Demo Optimization)

Creating student module for MIPRO...
⚠️  Using Predict (not ChainOfThought) for reliable optimization
  Training: 6 examples
  Validation: 2 examples

Optimizing... (3-5 minutes)
Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 17%|█▋        | 1/6 [00:05<00:28,  5.64s/it]

  Score: 3.50/5.0 (normalized: 0.625) → ✗ FAIL


 33%|███▎      | 2/6 [00:10<00:20,  5.21s/it]

...


Predictor 0
i: You are Farmer.CHAT, an experienced agricultural advisor dedicated to supporting smallholder farmers in Bihar, India. 

**YOUR ROLE:**
- Provide targeted, actionable advice based on farmers' specific questions and local farming conditions.
- Facilitate understanding by using a supportive and culturally appropriate communication style.

**RESPONSE GUIDELINES:**

1. **Content Quality:**
 - Directly address the farmer's query with clear, practical solutions.
 - Ensure advice is specific to Bihar's local conditions, incorporating regional crops and practices.
 - Offer insights on timing for agricultural activities relevant to the current season and crop growth stage.

2. **Communication Style:**
 - Maintain a warm and encouraging tone; communicate in a simple and conversational manner.
 - Clarify technical terms to ensure comprehension while avoiding formal jargon.

3. **Practical Advice:**
 - Emphasize low-cost, feasible solutions tailored for resource-limited farmers.
 - 

2026/01/10 17:47:30 INFO dspy.evaluate.evaluate: Average Metric: 2 / 2 (100.0%)
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 5'].
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [100.0, 100.0, 100.0, 50.0, 100.0, 100.0, 100.0, 100.0, 50.0, 100.0]
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 11 / 10 =====
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Evaluating the following candidate program...




Predictor 0
i: You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar, India.

**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region

**RESPONSE GUIDELINES:**

1. **Content Quality:**
 - Address the specific concern directly and practically
 - Provide actionable, region-appropriate advice
 - Include timing considerations based on current stage and season
 - Use local examples, varieties, and practices when relevant

2. **Communication Style:**
 - Warm, professional, and encouraging tone
 - Use simple, conversational language with appropriate cultural context
 - Explain technical concepts in simple terms
 - Avoid overly formal or academic language

3. **Practical Advice:**
 - Focus on low-cost, accessible solutions
 - Consider resource constraints of smallholder farmers
 - Mention local availability of 

2026/01/10 17:47:30 INFO dspy.evaluate.evaluate: Average Metric: 2 / 2 (100.0%)
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0'].
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [100.0, 100.0, 100.0, 50.0, 100.0, 100.0, 100.0, 100.0, 50.0, 100.0, 100.0]
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/01/10 17:47:30 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!




MIPRO OPTIMIZATION RESULTS

✓ Best score: 100.0
✓ Evaluated 11 candidate programs

Predictor type: Predict

📚 Few-Shot Demos: 0

  ⚠️  No demos in main result, searching 11 candidates...
    Candidate 1: 3 demos, instruction optimized
    Candidate 2: 0 demos, instruction optimized
    Candidate 3: 3 demos, instruction optimized
    Candidate 4: 3 demos, instruction baseline
    Candidate 5: 0 demos, instruction optimized
    Candidate 6: 3 demos, instruction optimized
    Candidate 7: 3 demos, instruction optimized
    Candidate 9: 3 demos, instruction optimized
    Candidate 10: 3 demos, instruction optimized

  ✓ Using candidate 1 instead of main result
  ✓ Now have 3 demos
  ✓ Successfully attached 3 demonstration examples

  Demo preview:
    1. Q: What fertilizer should I use for potato crops?...
       A: For your potato crops, a balanced NPK fertilizer ratio of 120:80:100 k...
    2. Q: How can I manage pests in my rice field effectively?...
       A: To manage pests effectiv

## 10. Extract and Compare Prompts

Now let's extract the actual prompts that each optimizer created and compare them.

**What we'll see:**
- The system instructions (docstring)
- Field descriptions
- Few-shot examples (if any)
- How they differ from the baseline

In [ ]:
from datetime import datetime
import json

def extract_prompt_configuration(module, module_name="Module"):
    """
    Extract the complete prompt configuration from a DSPy module.
    """
    print(f"\n{'='*80}")
    print(f"EXTRACTING: {module_name}")
    print(f"{'='*80}")

    config = {
        "module_name": module_name,
        "timestamp": datetime.now().isoformat()
    }

    try:
        # Access the predictor (ChainOfThought or Predict)
        predictor = module.generate_response

        # Get signature from inner predictor if ChainOfThought
        if hasattr(predictor, 'predictors'):
            # ChainOfThought wraps an inner Predict
            inner_predictors = list(predictor.predictors())
            if inner_predictors:
                inner_pred = inner_predictors[0]
                signature = inner_pred.signature
            else:
                signature = predictor.signature
        else:
            signature = predictor.signature

        # Get system prompt
        if hasattr(signature, 'instructions'):
            system_prompt = signature.instructions
        elif hasattr(signature, '__doc__'):
            system_prompt = signature.__doc__ or ""
        else:
            system_prompt = ""

        config['system_prompt'] = system_prompt

        # Get signature name
        sig_name = signature.__name__ if hasattr(signature, '__name__') else "Signature"
        config['signature_name'] = sig_name

        print(f"\n📋 Signature: {sig_name}")
        print(f"📝 System prompt length: {len(system_prompt)} characters")

        # Get input fields
        config['input_fields'] = {}
        if hasattr(signature, 'input_fields'):
            print(f"📥 Input fields: {', '.join(signature.input_fields.keys())}")
            for name, field in signature.input_fields.items():
                desc = getattr(field, 'desc', 'No description')
                config['input_fields'][name] = desc

        # Get output fields
        config['output_fields'] = {}
        if hasattr(signature, 'output_fields'):
            print(f"📤 Output fields: {', '.join(signature.output_fields.keys())}")
            for name, field in signature.output_fields.items():
                desc = getattr(field, 'desc', 'No description')
                config['output_fields'][name] = desc

        # Get demos
        demos = getattr(predictor, 'demos', [])
        config['num_demos'] = len(demos)
        config['demos'] = []

        if demos:
            print(f"📚 Few-shot demos: {len(demos)}")
            for demo in demos:
                demo_dict = {
                    'question': demo.farmer_question if hasattr(demo, 'farmer_question') else '',
                    'response': demo.response if hasattr(demo, 'response') else ''
                }
                config['demos'].append(demo_dict)
        else:
            print(f"📚 Few-shot demos: 0")

        return config

    except Exception as e:
        print(f"\n⚠️  Error extracting configuration: {e}")
        return {
            "module_name": module_name,
            "error": str(e),
            "timestamp": datetime.now().isoformat()
        }

# Extract configurations from all models
baseline_config = extract_prompt_configuration(baseline_module, "Baseline (Original)")
bootstrap_config = extract_prompt_configuration(optimized_bootstrap, "BootstrapFewShot")
random_search_config = extract_prompt_configuration(optimized_random_search, "RandomSearch")
mipro_config = extract_prompt_configuration(optimized_mipro, "MIPRO") # Rerun MIPRO cell -> should work

print("\n✓ All configurations extracted!")


EXTRACTING: Baseline (Original)

📋 Signature: StringSignature
📝 System prompt length: 2050 characters
📥 Input fields: farmer_question, agricultural_facts, crop_context
📤 Output fields: reasoning, response
📚 Few-shot demos: 0

EXTRACTING: BootstrapFewShot

📋 Signature: FactStitching
📝 System prompt length: 2050 characters
📥 Input fields: farmer_question, agricultural_facts, crop_context
📤 Output fields: response
📚 Few-shot demos: 8

EXTRACTING: RandomSearch

📋 Signature: StringSignature
📝 System prompt length: 2050 characters
📥 Input fields: farmer_question, agricultural_facts, crop_context
📤 Output fields: reasoning, response
📚 Few-shot demos: 0

EXTRACTING: MIPRO

📋 Signature: StringSignature
📝 System prompt length: 582 characters
📥 Input fields: farmer_question, agricultural_facts, crop_context
📤 Output fields: response
📚 Few-shot demos: 3

✓ All configurations extracted!


## 11. Print Complete Prompts

Let's print the full system prompts for each optimizer so you can see exactly what changed.

In [ ]:
def print_full_prompt(config):
    """
    Print the complete prompt configuration in a readable format.
    """
    print(f"\n{'='*80}")
    print(f"FULL PROMPT: {config['module_name']}")
    print(f"{'='*80}")

    if 'error' in config:
        print(f"\n⚠️  Error: {config['error']}")
        return

    print(f"\n{'─'*80}")
    print("SYSTEM INSTRUCTIONS:")
    print(f"{'─'*80}")
    print(config['system_prompt'])

    print(f"\n{'─'*80}")
    print("INPUT FIELDS:")
    print(f"{'─'*80}")
    for field_name, desc in config['input_fields'].items():
        print(f"\n{field_name}:")
        print(f"  {desc}")

    print(f"\n{'─'*80}")
    print("OUTPUT FIELDS:")
    print(f"{'─'*80}")
    for field_name, desc in config['output_fields'].items():
        print(f"\n{field_name}:")
        print(f"  {desc}")

    if config['num_demos'] > 0:
        print(f"\n{'─'*80}")
        print(f"FEW-SHOT EXAMPLES: {config['num_demos']} demos")
        print(f"{'─'*80}")
        for i, demo in enumerate(config['demos'][:3], 1):  # Show first 3
            print(f"\nExample {i}:")
            print(f"  Q: {demo['question'][:100]}...")
            print(f"  A: {demo['response'][:100]}...")

        if config['num_demos'] > 3:
            print(f"\n... and {config['num_demos'] - 3} more examples")

    print(f"\n{'='*80}\n")

# Print all prompts
print_full_prompt(baseline_config)
print_full_prompt(bootstrap_config)
print_full_prompt(random_search_config)
print_full_prompt(mipro_config)

# =============================================================================
# CELL 12: Save Prompts to Files
# =============================================================================

def save_prompt_to_file(config, filename):
    """
    Save prompt configuration to a text file.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"{'='*80}\n")
        f.write(f"OPTIMIZED PROMPT: {config['module_name']}\n")
        f.write(f"Generated: {config['timestamp']}\n")
        f.write(f"{'='*80}\n\n")

        if 'error' in config:
            f.write(f"⚠️  Error: {config['error']}\n")
            return

        f.write(f"SYSTEM INSTRUCTIONS:\n")
        f.write(f"{'-'*80}\n")
        f.write(config['system_prompt'])
        f.write(f"\n\n")

        f.write(f"INPUT FIELDS:\n")
        f.write(f"{'-'*80}\n")
        for field_name, desc in config['input_fields'].items():
            f.write(f"\n{field_name}: {desc}\n")

        f.write(f"\nOUTPUT FIELDS:\n")
        f.write(f"{'-'*80}\n")
        for field_name, desc in config['output_fields'].items():
            f.write(f"\n{field_name}: {desc}\n")

        if config['num_demos'] > 0:
            f.write(f"\nFEW-SHOT EXAMPLES: {config['num_demos']} demonstrations\n")
            f.write(f"{'-'*80}\n")
            for i, demo in enumerate(config['demos'], 1):
                f.write(f"\nExample {i}:\n")
                f.write(f"  Q: {demo['question']}\n")
                f.write(f"  A: {demo['response']}\n")

    print(f"✓ Saved: {filename}")

# Save all prompts
print("\nSaving prompts to files...\n")
save_prompt_to_file(baseline_config, "prompt_baseline.txt")
save_prompt_to_file(bootstrap_config, "prompt_bootstrap_fewshot.txt")
save_prompt_to_file(random_search_config, "prompt_random_search.txt")
save_prompt_to_file(mipro_config, "prompt_mipro.txt")

print("\n✓ All prompts saved!")


FULL PROMPT: Baseline (Original)

────────────────────────────────────────────────────────────────────────────────
SYSTEM INSTRUCTIONS:
────────────────────────────────────────────────────────────────────────────────
You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar, India.

**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region

**RESPONSE GUIDELINES:**

1. **Content Quality:**
 - Address the specific concern directly and practically
 - Provide actionable, region-appropriate advice
 - Include timing considerations based on current stage and season
 - Use local examples, varieties, and practices when relevant

2. **Communication Style:**
 - Warm, professional, and encouraging tone
 - Use simple, conversational language with appropriate cultural context
 - Explain technical concepts in simple terms


## 12. Save Prompts to Files

Save each optimized prompt to a text file for easy sharing and version control.

In [ ]:
def save_prompt_to_file(config, filename):
    """
    Save prompt configuration to a text file.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"{'='*80}\n")
        f.write(f"OPTIMIZED PROMPT: {config['module_name']}\n")
        f.write(f"Generated: {config['timestamp']}\n")
        f.write(f"{'='*80}\n\n")

        f.write(f"SYSTEM INSTRUCTIONS:\n")
        f.write(f"{'-'*80}\n")
        f.write(config['system_prompt'])
        f.write(f"\n\n")

        f.write(f"INPUT FIELDS:\n")
        f.write(f"{'-'*80}\n")
        for field_name, desc in config['input_fields'].items():
            f.write(f"\n{field_name}: {desc}\n")

        f.write(f"\nOUTPUT FIELDS:\n")
        f.write(f"{'-'*80}\n")
        for field_name, desc in config['output_fields'].items():
            f.write(f"\n{field_name}: {desc}\n")

        if config['num_demos'] > 0:
            f.write(f"\nFEW-SHOT EXAMPLES: {config['num_demos']} demonstrations\n")
            f.write(f"{'-'*80}\n")
            for i, demo in enumerate(config['demos'], 1):
                f.write(f"\nExample {i}:\n")
                f.write(f"  Q: {demo['question']}\n")
                f.write(f"  A: {demo['response']}\n")

    print(f"✓ Saved: {filename}")

# Save all prompts
print("\nSaving prompts to files...\n")
save_prompt_to_file(baseline_config, "prompt_baseline.txt")
save_prompt_to_file(bootstrap_config, "prompt_bootstrap_fewshot.txt")
save_prompt_to_file(random_search_config, "prompt_random_search.txt")
save_prompt_to_file(mipro_config, "prompt_mipro.txt")

print("\n✓ All prompts saved!")


Saving prompts to files...

✓ Saved: prompt_baseline.txt
✓ Saved: prompt_bootstrap_fewshot.txt
✓ Saved: prompt_random_search.txt
✓ Saved: prompt_mipro.txt

✓ All prompts saved!


## 13. Compare Performance

Let's test all optimized models on a few examples and compare their performance.

In [ ]:
# =============================================================================
# CELL 13: Performance Comparison
# =============================================================================

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)

# Test on a few examples
test_examples = trainset[:3]

# Build models dict
models = {
    "Baseline": baseline_module,
    "BootstrapFewShot": optimized_bootstrap,
    "RandomSearch": optimized_random_search,
    "MIPRO": optimized_mipro
}

print(f"\nTesting {len(models)} models on {len(test_examples)} examples\n")

results = {name: [] for name in models.keys()}

for i, example in enumerate(test_examples, 1):
    print(f"\n{'─'*80}")
    print(f"Test Example {i}: {example.farmer_question[:60]}...")
    print(f"{'─'*80}")

    for model_name, model in models.items():
        try:
            # Generate prediction
            pred = model(
                farmer_question=example.farmer_question,
                agricultural_facts=example.agricultural_facts,
                crop_context=example.crop_context
            )

            # Evaluate with full score (not boolean)
            raw_score = evaluator.get_overall_score(example.farmer_question, pred.response)
            normalized = (raw_score - 1) / 4
            results[model_name].append(normalized)

            print(f"\n{model_name}: {raw_score:.2f}/5.0")
            print(f"  Response: {pred.response[:120]}...")
        except Exception as e:
            print(f"\n{model_name}: ERROR - {e}")
            results[model_name].append(0.5)

# Print summary
print(f"\n{'='*80}")
print("AVERAGE SCORES ACROSS ALL TEST EXAMPLES")
print(f"{'='*80}\n")

for model_name, scores in results.items():
    avg_score = sum(scores) / len(scores) if scores else 0
    raw_avg = (avg_score * 4) + 1  # Convert back to 1-5 scale
    print(f"{model_name:35s} {raw_avg:.2f}/5.0  (normalized: {avg_score:.3f})")

# Find best model
best_model = max(results.items(), key=lambda x: sum(x[1])/len(x[1]) if x[1] else 0)
best_avg = sum(best_model[1])/len(best_model[1]) if best_model[1] else 0
best_raw = (best_avg * 4) + 1

print(f"\n🏆 Best performing model: {best_model[0]}")
print(f"   Average score: {best_raw:.2f}/5.0")


PERFORMANCE COMPARISON

Testing 4 models on 3 examples


────────────────────────────────────────────────────────────────────────────────
Test Example 1: How can I manage pests in my rice field effectively?...
────────────────────────────────────────────────────────────────────────────────

Baseline: 4.50/5.0
  Response: To manage pests in your rice field effectively, start by identifying the brown planthopper, a common pest that can harm ...

BootstrapFewShot: 4.30/5.0
  Response: To effectively manage pests in your rice field, pay special attention to the brown planthopper, which is a common issue ...

RandomSearch: 4.50/5.0
  Response: To manage pests in your rice field effectively, start by identifying the brown planthopper, a common pest that can harm ...

MIPRO: 4.50/5.0
  Response: To effectively manage pests in your rice field, it's important to keep an eye out for the brown planthopper, a common pe...

──────────────────────────────────────────────────────────────────────────

## 14. Save All Results to JSON

Save all configurations and results for later analysis.

In [ ]:
# =============================================================================
# CELL 14: Save All Results to JSON
# =============================================================================

# Compile all results
final_results = {
    "timestamp": datetime.now().isoformat(),
    "baseline": baseline_config,
    "bootstrap_fewshot": bootstrap_config,
    "random_search": random_search_config,
    "mipro": mipro_config,
    "performance_scores": results,
    "average_scores": {
        name: {
            "normalized": sum(scores) / len(scores) if scores else 0,
            "raw_5_scale": ((sum(scores) / len(scores)) * 4 + 1) if scores else 1.0
        }
        for name, scores in results.items()
    },
    "best_model": {
        "name": best_model[0],
        "score": best_raw,
        "normalized": best_avg
    }
}

# Save to JSON
output_filename = f"optimization_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print(f"\n{'='*80}")
print("RESULTS SAVED")
print(f"{'='*80}")
print(f"\n✓ Results saved to: {output_filename}")
print(f"  File size: {len(json.dumps(final_results)) / 1024:.1f} KB")

print(f"\n✓ Text files saved:")
print(f"  - prompt_baseline.txt")
print(f"  - prompt_bootstrap_fewshot.txt")
print(f"  - prompt_random_search.txt")
print(f"  - prompt_mipro.txt")

print(f"\n{'='*80}")
print("OPTIMIZATION COMPLETE!")
print(f"{'='*80}")

## 15. Summary and Next Steps

### What We Did:
1. ✅ Defined a baseline prompt (FactStitching signature)
2. ✅ Created synthetic training examples
3. ✅ Defined an evaluation metric
4. ✅ Ran three different optimizers:
   - BootstrapFewShot (adds examples)
   - BootstrapFewShotWithRandomSearch (adds examples + instruction variations)
   - MIPRO (advanced multi-prompt optimization)
5. ✅ Extracted and compared optimized prompts
6. ✅ Saved prompts to files for easy sharing
7. ✅ Evaluated performance across models

### Key Takeaways:
- **BootstrapFewShot** is fast and effective when good examples exist
- **RandomSearch** explores instruction variations automatically
- **MIPRO** is the most advanced but computationally expensive
- Optimized prompts can be extracted and reused
- Performance improvement varies by task and data quality

### Next Steps:
1. **Replace the example task** with your own domain
2. **Create better training examples** specific to your use case
3. **Refine the evaluation metric** to match your quality criteria
4. **Experiment with different optimizers** and parameters
5. **A/B test** the optimized prompts in production

### Tips for Your Own Tasks:
- Start with 5-10 high-quality training examples
- Make your metric specific to what matters for your task
- Try BootstrapFewShot first (fastest)
- Use MIPRO when you need maximum performance
- Always extract and review optimized prompts before deploying

---

## Running TEST CELL: Best way to extract optimized prompt

### Technical Discovery: DSPy Optimizer Storage Architecture

Through systematic testing of 6 extraction methods across 3 optimizers, it was discovered:

**BootstrapFewShot (Works Correctly)**
- ✅ Directly applies demos to returned module
- ✅ Standard attribute access works
- ✅ No extraction workarounds needed

**RandomSearch & MIPRO (Require Extraction)**  
- ❌ Store optimizations in `candidate_programs` metadata
- ❌ Don't apply to returned module by default
- ✅ Extraction from `candidate_programs` required

**Verification Methods (All Optimizers)**
- `dspy.inspect_history(n=1)` - Shows actual LLM prompt (100% reliable)
- `model.save("file.json")` - Exports complete state (100% reliable)
- Direct attribute access - Works only for BootstrapFewShot (33% reliable)

**Conclusion:** So far we have needed to use `candidate_programs` extraction to handle DSPy's architecture, it is not a workaround.

Down below is the running test cell, please feel free to use it as context for further experimentation

In [ ]:
# =============================================================================
# TEST CELL: Compare Extraction Methods - Our Way vs "Official" DSPy Way
# =============================================================================

print("\n" + "="*90)
print("TESTING: EXTRACTION METHODS COMPARISON")
print("="*90)

def test_optimizer_extraction(optimizer_name, optimized_model, test_example):
    """
    Test different methods to extract optimized prompts and compare results.
    """
    print(f"\n{'─'*90}")
    print(f"TESTING: {optimizer_name}")
    print(f"{'─'*90}")

    results = {
        "optimizer": optimizer_name,
        "methods": {}
    }

    # =========================================================================
    # METHOD 1: Our Manual Extraction (Current Approach)
    # =========================================================================
    print(f"\n📋 METHOD 1: Our Manual Extraction")
    print(f"{'·'*90}")

    try:
        predictor = optimized_model.generate_response

        # Handle ChainOfThought wrapper
        if hasattr(predictor, 'predictors'):
            inner_preds = list(predictor.predictors())
            if inner_preds:
                sig = inner_preds[0].signature
            else:
                sig = predictor.signature
        else:
            sig = predictor.signature

        # Get instructions
        instructions = getattr(sig, 'instructions', None) or getattr(sig, '__doc__', '')

        # Get demos
        demos = getattr(predictor, 'demos', [])

        results["methods"]["manual_extraction"] = {
            "success": True,
            "instruction_length": len(instructions),
            "num_demos": len(demos),
            "instruction_preview": instructions[:100] if instructions else "None"
        }

        print(f"✓ Instructions: {len(instructions)} chars")
        print(f"✓ Demos: {len(demos)}")
        print(f"  Preview: {instructions[:100]}..." if instructions else "  No instructions")

    except Exception as e:
        results["methods"]["manual_extraction"] = {"success": False, "error": str(e)}
        print(f"✗ Failed: {e}")

    # =========================================================================
    # METHOD 2: Direct Signature Access (Suggested Method)
    # =========================================================================
    print(f"\n📋 METHOD 2: Direct Signature Access (compiled_program.predictor.signature)")
    print(f"{'·'*90}")

    try:
        # Try accessing predictor directly
        if hasattr(optimized_model, 'generate_response'):
            predictor = optimized_model.generate_response

            # Try direct signature access
            sig = predictor.signature
            instructions = getattr(sig, 'instructions', None)

            if instructions:
                results["methods"]["direct_signature"] = {
                    "success": True,
                    "instruction_length": len(instructions),
                    "instruction_preview": instructions[:100]
                }
                print(f"✓ Instructions found: {len(instructions)} chars")
                print(f"  Preview: {instructions[:100]}...")
            else:
                results["methods"]["direct_signature"] = {"success": False, "reason": "No instructions attribute"}
                print(f"✗ No instructions attribute on signature")
        else:
            results["methods"]["direct_signature"] = {"success": False, "reason": "No generate_response"}
            print(f"✗ No generate_response attribute")

    except Exception as e:
        results["methods"]["direct_signature"] = {"success": False, "error": str(e)}
        print(f"✗ Failed: {e}")

    # =========================================================================
    # METHOD 3: Demos Access (Few-Shot Examples)
    # =========================================================================
    print(f"\n📋 METHOD 3: Demos Access (compiled_program.predictor.demos)")
    print(f"{'·'*90}")

    try:
        predictor = optimized_model.generate_response
        demos = getattr(predictor, 'demos', None)

        if demos and len(demos) > 0:
            results["methods"]["demos_access"] = {
                "success": True,
                "num_demos": len(demos),
                "demo_preview": str(demos[0])[:100] if demos else None
            }
            print(f"✓ Demos found: {len(demos)}")
            print(f"  First demo: {str(demos[0])[:100]}...")
        else:
            results["methods"]["demos_access"] = {"success": False, "reason": "No demos"}
            print(f"✗ No demos found (demos = {demos})")

    except Exception as e:
        results["methods"]["demos_access"] = {"success": False, "error": str(e)}
        print(f"✗ Failed: {e}")

    # =========================================================================
    # METHOD 4: inspect_history (Run Then Inspect)
    # =========================================================================
    print(f"\n📋 METHOD 4: dspy.inspect_history (Run model first)")
    print(f"{'·'*90}")

    try:
        # Run the model once to populate history
        print(f"  Running model on test example...")
        _ = optimized_model(
            farmer_question=test_example.farmer_question,
            agricultural_facts=test_example.agricultural_facts,
            crop_context=test_example.crop_context
        )

        # Try to inspect history
        print(f"  Attempting dspy.inspect_history(n=1)...")

        # Capture inspect_history output
        import io
        from contextlib import redirect_stdout

        f = io.StringIO()
        with redirect_stdout(f):
            dspy.inspect_history(n=1)

        history_output = f.getvalue()

        if history_output and len(history_output) > 100:
            results["methods"]["inspect_history"] = {
                "success": True,
                "output_length": len(history_output),
                "contains_instructions": "Farmer.CHAT" in history_output or "agricultural" in history_output,
                "preview": history_output[:200]
            }
            print(f"✓ History captured: {len(history_output)} chars")
            print(f"  Contains prompt: {'Yes' if 'Farmer' in history_output else 'No'}")
            print(f"  Preview:\n{history_output[:300]}...")
        else:
            results["methods"]["inspect_history"] = {"success": False, "reason": "Empty or short output"}
            print(f"✗ History output too short or empty")

    except Exception as e:
        results["methods"]["inspect_history"] = {"success": False, "error": str(e)}
        print(f"✗ Failed: {e}")

    # =========================================================================
    # METHOD 5: Save and Load (Export to JSON)
    # =========================================================================
    print(f"\n📋 METHOD 5: Save to JSON (compiled_program.save)")
    print(f"{'·'*90}")

    try:
        import tempfile
        import os

        # Create temp file
        temp_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
        temp_path = temp_file.name
        temp_file.close()

        print(f"  Saving to: {temp_path}")
        optimized_model.save(temp_path)

        # Read back
        with open(temp_path, 'r') as f:
            saved_content = f.read()

        # Clean up
        os.unlink(temp_path)

        if len(saved_content) > 100:
            results["methods"]["save_json"] = {
                "success": True,
                "file_size": len(saved_content),
                "contains_instructions": "instructions" in saved_content,
                "contains_demos": "demos" in saved_content
            }
            print(f"✓ Saved successfully: {len(saved_content)} bytes")
            print(f"  Contains 'instructions': {'Yes' if 'instructions' in saved_content else 'No'}")
            print(f"  Contains 'demos': {'Yes' if 'demos' in saved_content else 'No'}")
        else:
            results["methods"]["save_json"] = {"success": False, "reason": "File too small"}
            print(f"✗ Saved file too small")

    except Exception as e:
        results["methods"]["save_json"] = {"success": False, "error": str(e)}
        print(f"✗ Failed: {e}")

    # =========================================================================
    # METHOD 6: candidate_programs (Our Discovery)
    # =========================================================================
    print(f"\n📋 METHOD 6: candidate_programs Extraction (Our Discovery)")
    print(f"{'·'*90}")

    try:
        if hasattr(optimized_model, 'candidate_programs') and optimized_model.candidate_programs:
            print(f"✓ Found {len(optimized_model.candidate_programs)} candidates")

            # Check first candidate
            if isinstance(optimized_model.candidate_programs[0], dict):
                if 'program' in optimized_model.candidate_programs[0]:
                    candidate = optimized_model.candidate_programs[0]['program']
                    cand_demos = getattr(candidate.generate_response, 'demos', [])

                    results["methods"]["candidate_programs"] = {
                        "success": True,
                        "num_candidates": len(optimized_model.candidate_programs),
                        "first_candidate_demos": len(cand_demos)
                    }
                    print(f"  First candidate has {len(cand_demos)} demos")
                else:
                    results["methods"]["candidate_programs"] = {"success": False, "reason": "No 'program' key"}
                    print(f"✗ No 'program' key in candidate dict")
            else:
                results["methods"]["candidate_programs"] = {"success": False, "reason": "Not dict format"}
                print(f"✗ Candidates not in dict format")
        else:
            results["methods"]["candidate_programs"] = {"success": False, "reason": "No candidate_programs attribute"}
            print(f"ℹ️  No candidate_programs (this optimizer may not use them)")

    except Exception as e:
        results["methods"]["candidate_programs"] = {"success": False, "error": str(e)}
        print(f"✗ Failed: {e}")

    return results

# =============================================================================
# Run Tests on All Optimizers
# =============================================================================

test_example = trainset[0]

all_results = {}

print("\n" + "="*90)
print("RUNNING EXTRACTION TESTS ON ALL OPTIMIZERS")
print("="*90)

# Test BootstrapFewShot
all_results["BootstrapFewShot"] = test_optimizer_extraction(
    "BootstrapFewShot",
    optimized_bootstrap,
    test_example
)

# Test RandomSearch
all_results["RandomSearch"] = test_optimizer_extraction(
    "RandomSearch",
    optimized_random_search,
    test_example
)

# Test MIPRO
all_results["MIPRO"] = test_optimizer_extraction(
    "MIPRO",
    optimized_mipro,
    test_example
)

# =============================================================================
# Summary Comparison
# =============================================================================

print("\n" + "="*90)
print("SUMMARY: METHOD SUCCESS RATES")
print("="*90)

methods = ["manual_extraction", "direct_signature", "demos_access",
           "inspect_history", "save_json", "candidate_programs"]

print(f"\n{'Method':<30} {'Bootstrap':<15} {'RandomSearch':<15} {'MIPRO':<15}")
print("─" * 90)

for method in methods:
    row = f"{method:<30}"
    for opt_name in ["BootstrapFewShot", "RandomSearch", "MIPRO"]:
        result = all_results[opt_name]["methods"].get(method, {})
        success = "✓" if result.get("success") else "✗"
        row += f" {success:<14}"
    print(row)

print("\n" + "="*90)
print("CONCLUSION")
print("="*90)

# Count successes
method_scores = {}
for method in methods:
    score = sum(
        1 for opt_name in all_results
        if all_results[opt_name]["methods"].get(method, {}).get("success")
    )
    method_scores[method] = score

best_method = max(method_scores.items(), key=lambda x: x[1])

print(f"\n🏆 Most Reliable Method: {best_method[0]}")
print(f"   Success rate: {best_method[1]}/3 optimizers")

print(f"\n📊 All Method Scores:")
for method, score in sorted(method_scores.items(), key=lambda x: -x[1]):
    print(f"   {method:<30} {score}/3 optimizers")

print("\n" + "="*90)


TESTING: EXTRACTION METHODS COMPARISON

RUNNING EXTRACTION TESTS ON ALL OPTIMIZERS

──────────────────────────────────────────────────────────────────────────────────────────
TESTING: BootstrapFewShot
──────────────────────────────────────────────────────────────────────────────────────────

📋 METHOD 1: Our Manual Extraction
··························································································
✓ Instructions: 2050 chars
✓ Demos: 8
  Preview: You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar, India.

**YOUR R...

📋 METHOD 2: Direct Signature Access (compiled_program.predictor.signature)
··························································································
✓ Instructions found: 2050 chars
  Preview: You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar, India.

**YOUR R...

📋 METHOD 3: Demos Access (compiled_program.predictor.demos)
····················································